# Dua kondisi YOLO11n: citra asli bersih vs baseline HE + brightness
Hanya dua kondisi yang dibandingkan. **Patience 15**, input 640, batch 8, maksimal 80 epoch, AdamW dan seed 42 sama dengan baseline notebook 02. Label tetap bersih: 252 citra, 3.342 anotasi, tanpa FDI berulang. Split 175/50/27 tetap sama.

Notebook ini melatih ulang **citra asli tanpa HE dan augmentasi**. Hasil baseline HE + brightness diambil dari run notebook 02 yang sudah selesai, sehingga tidak perlu mengulang training baseline. Data awal lama ber-patience 20 tidak dipakai. Buka GPU T4 lalu Run all.

Perbandingan menguji pengaruh paket preprocessing dan augmentasi bersama-sama. Pengaruh HE dan brightness secara terpisah tidak disimpulkan. Train asli berisi 175 file dan train baseline 525 file. Anggaran epoch sama, tetapi jumlah optimizer steps berbeda.


In [1]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.3.221','scipy','pandas','pyyaml','pillow','opencv-python-headless'])
import torch, yaml, cv2
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Image as DisplayImage
if not torch.cuda.is_available():
    raise RuntimeError('Pilih Runtime > Change runtime type > T4 GPU, kemudian Run all.')
print('GPU:',torch.cuda.get_device_name(0))


GPU: Tesla T4


In [3]:
# Lokasi Drive sesuai folder yang diberikan pengguna.
from pathlib import Path, PurePosixPath
import os, json, hashlib, unicodedata, zipfile, shutil

PROJECT_NAME = '1226800011_Aprilia_UjianEDA_Revisi'
YOLO_FOLDER = 'enumerasi gigi susu6-252-.v1i.yolov11'
YOLO_ZIP = 'enumerasi_gigi_susu_252_label_asli.zip'

def normal_name(value):
    return ' '.join(unicodedata.normalize('NFKC', value).split()).casefold()

def get_project_root():
    try:
        from google.colab import drive
    except ImportError:
        # Hanya untuk verifikasi lokal. Di Colab, lokasi default tetap /content/drive.
        pass
    else:
        drive.mount('/content/drive')
    mydrive = Path(os.environ.get('FDI_MYDRIVE', '/content/drive/MyDrive'))
    direct = mydrive / PROJECT_NAME
    if direct.is_dir():
        return direct
    if not mydrive.is_dir():
        raise FileNotFoundError('Drive belum tersedia. Jalankan notebook ini di Google Colab dan izinkan akses Drive.')
    matches = [p for p in mydrive.iterdir()
               if p.is_dir() and normal_name(p.name) == normal_name(PROJECT_NAME)]
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Folder {PROJECT_NAME!r} belum ditemukan secara unik pada {mydrive}. '
            'Pastikan akun saat mount Drive adalah akun pemilik folder tersebut. '
            f'Kandidat: {matches}'
        )
    return matches[0]

def yolo_is_complete(root):
    images = root / 'train' / 'images'
    labels = root / 'train' / 'labels'
    if not ((root / 'data.yaml').is_file() and images.is_dir() and labels.is_dir()):
        return False
    paths = [p for p in images.iterdir()
             if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}]
    return len(paths) == 252 and all((labels / (p.stem + '.txt')).is_file() for p in paths)

def get_raw_yolo(project_root):
    unpacked = project_root / YOLO_FOLDER
    if yolo_is_complete(unpacked):
        print('Ekspor YOLO asli:', unpacked)
        return unpacked
    archive = project_root / YOLO_ZIP
    if not archive.is_file():
        raise FileNotFoundError(
            f'Ekspor YOLO dengan label FDI belum tersedia. Letakkan {YOLO_ZIP} '
            f'atau folder {YOLO_FOLDER} pada {project_root}. '
            'Empat folder sumber saja belum memuat label FDI yang diperlukan.'
        )
    digest = hashlib.sha256()
    with archive.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    scratch = Path(os.environ.get('FDI_SCRATCH', '/content'))
    extracted = scratch / ('fdi_raw_' + digest.hexdigest()[:12])
    root = extracted / YOLO_FOLDER
    if not yolo_is_complete(root):
        extracted.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            if z.testzip() is not None:
                raise RuntimeError('ZIP belum lengkap atau rusak. Tunggu unggahan selesai lalu jalankan ulang.')
            for info in z.infolist():
                rel = PurePosixPath(info.filename)
                if rel.is_absolute() or '..' in rel.parts or chr(92) in info.filename:
                    raise ValueError('ZIP memiliki lokasi file yang tidak aman.')
                if ((info.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError('ZIP dengan symbolic link tidak digunakan.')
                target = (extracted / Path(*rel.parts)).resolve()
                if not target.is_relative_to(extracted.resolve()):
                    raise ValueError('Lokasi ekstraksi keluar dari folder tujuan.')
            z.extractall(extracted)
    if not yolo_is_complete(root):
        raise RuntimeError('Ekspor YOLO harus memuat data.yaml, 252 citra dan pasangan file label aslinya.')
    print('ZIP YOLO diekstrak otomatis ke penyimpanan lokal runtime:', root)
    return root

PROJECT_ROOT = get_project_root()
print('Folder proyek Drive:', PROJECT_ROOT)

PROJECT = PROJECT_ROOT / 'hasil_analisis'
SOURCE = PROJECT / 'dataset_terkoreksi'
completion_file = PROJECT / 'RUN_COMPLETE.json'
if not completion_file.is_file():
    print('Hasil notebook 01 belum tersedia. Menjalankan persiapan data otomatis terlebih dahulu...')
    first_notebook = PROJECT_ROOT / '01_EDA_Penanganan_Hasil_Akhir.ipynb'
    if not first_notebook.is_file():
        raise FileNotFoundError('Notebook 01 harus berada pada folder proyek yang sama.')
    from IPython.display import display
    first = json.loads(first_notebook.read_text(encoding='utf-8'))
    namespace = {'__name__':'auto_prepare_fdi','display':display}
    for number,cell in enumerate(first['cells'],1):
        if cell['cell_type']=='code':
            print(f'Persiapan data otomatis: cell {number}')
            exec(compile(''.join(cell['source']),f'auto_EDA_cell_{number}','exec'),namespace)
if not completion_file.is_file() or not (SOURCE/'manifest.csv').is_file():
    raise RuntimeError('Persiapan data belum selesai. Jalankan notebook 01 sampai cell terakhir.')
completion = json.loads(completion_file.read_text(encoding='utf-8'))
actual = hashlib.sha256((SOURCE/'manifest.csv').read_bytes()).hexdigest()
if completion.get('status')!='complete' or completion.get('manifest_sha256')!=actual:
    raise RuntimeError('Manifest hasil berubah atau proses belum lengkap. Jalankan ulang notebook 01.')
print('Dataset terkoreksi siap:',SOURCE)

RAW_YOLO=get_raw_yolo(PROJECT_ROOT)
from datetime import datetime
from zoneinfo import ZoneInfo
RUN_ID=datetime.now(ZoneInfo('Asia/Jakarta')).strftime('%Y%m%d_%H%M%S')
ABLATION_HOME=PROJECT_ROOT/'hasil_perbandingan_dua_YOLO11n'/RUN_ID
ABLATION_HOME.mkdir(parents=True,exist_ok=False)
LOCAL_HOME=Path(os.environ.get('FDI_SCRATCH','/content'))/f'fdi_ablasi_{RUN_ID}'
LOCAL_HOME.mkdir(parents=True,exist_ok=False)
original=pd.read_csv(PROJECT/'tabel'/'manifest_penanganan.csv')
labels=pd.read_csv(PROJECT/'tabel'/'anotasi_terkoreksi.csv')
names=yaml.safe_load((RAW_YOLO/'data.yaml').read_text(encoding='utf-8-sig'))['names']
FDI=[str(names[i]) for i in range(len(names))] if isinstance(names,dict) else list(map(str,names))
assert len(original)==252 and original.image_id.nunique()==252
assert len(labels)==3342
assert not labels.duplicated(['image_id','class_id']).any(), 'Masih ada FDI berulang.'
assert original.groupby('group_id').split.nunique().max()==1
assert original.split.value_counts().to_dict()=={'train':175,'valid':50,'test':27}
original.to_csv(ABLATION_HOME/'manifest_sumber_bersih.csv',index=False)
labels.to_csv(ABLATION_HOME/'anotasi_bersih.csv',index=False)
print('Sumber label sudah bersih. Hasil:',ABLATION_HOME)

BASELINE_RUN=PROJECT/'pemodelan'/'20261031_124203'
baseline_meta=json.loads((BASELINE_RUN/'baseline_metadata.json').read_text())
assert baseline_meta['training_config']['patience']==15
baseline_frame=pd.read_csv(BASELINE_RUN/'perbandingan_test.csv')
BASELINE_RESULT=baseline_frame[baseline_frame.model=='baseline_yolo11n'].iloc[0].to_dict()
print('Baseline HE + brightness tersedia. Patience:',baseline_meta['training_config']['patience'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder proyek Drive: /content/drive/MyDrive/1226800011_Aprilia_UjianEDA_Revisi
Dataset terkoreksi siap: /content/drive/MyDrive/1226800011_Aprilia_UjianEDA_Revisi/hasil_analisis/dataset_terkoreksi
ZIP YOLO diekstrak otomatis ke penyimpanan lokal runtime: /content/fdi_raw_fd36b5351138/enumerasi gigi susu6-252-.v1i.yolov11
Sumber label sudah bersih. Hasil: /content/drive/MyDrive/1226800011_Aprilia_UjianEDA_Revisi/hasil_perbandingan_dua_YOLO11n/20261031_173820
Baseline HE + brightness tersedia. Patience: 15


## Siapkan hanya citra asli bersih, tanpa HE dan tanpa augmentasi

In [4]:
SCENARIOS={'A0_asli_bersih': {'he':False,'augment':False}}

def orient_point(x,y,o):
    return {1:(x,y),2:(1-x,y),3:(1-x,1-y),4:(x,1-y),5:(y,x),6:(1-y,x),7:(1-y,1-x),8:(y,1-x)}[o]

def raw_label_text(frame,orientation):
    # CSV koreksi sudah memakai koordinat setelah EXIF. Balikkan ke posisi piksel asli.
    inverse={1:1,2:2,3:3,4:4,5:5,6:8,7:7,8:6}[orientation]
    lines=[]
    for b in frame.itertuples():
        points=[orient_point(x,y,inverse) for x in [b.x-b.w/2,b.x+b.w/2] for y in [b.y-b.h/2,b.y+b.h/2]]
        xs,ys=zip(*points)
        x,y=(min(xs)+max(xs))/2,(min(ys)+max(ys))/2
        w,h=max(xs)-min(xs),max(ys)-min(ys)
        assert 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
        lines.append(f'{int(b.class_id)} {x:.8f} {y:.8f} {w:.8f} {h:.8f}')
    return '\n'.join(lines)

DATASETS={};dataset_rows=[]
for scenario,setting in SCENARIOS.items():
    root=LOCAL_HOME/scenario
    for split in ['train','valid','test']:
        (root/split/'images').mkdir(parents=True)
        (root/split/'labels').mkdir()
    manifest=[]
    for i,r in enumerate(original.itertuples(),1):
        p=RAW_YOLO/'train'/'images'/r.filename
        assert hashlib.sha256(p.read_bytes()).hexdigest()==r.file_sha256, f'Citra sumber berubah: {r.image_id}'
        with Image.open(p) as im:
            orientation=int(im.getexif().get(274,1))
            gray=np.asarray(im.convert('L')).copy()
        text=raw_label_text(labels[labels.image_id==r.image_id],orientation)
        if setting['he']:gray=cv2.equalizeHist(gray)
        variants=[('base',1.)]
        if r.split=='train' and setting['augment']:variants += [('dark',.85),('bright',1.15)]
        for suffix,factor in variants:
            arr=np.clip(gray.astype(np.float32)*factor,0,255).astype(np.uint8)
            stem=f'{r.image_id}_{suffix}'
            Image.fromarray(arr).convert('RGB').save(root/r.split/'images'/f'{stem}.png')
            (root/r.split/'labels'/f'{stem}.txt').write_text(text,encoding='utf-8')
            pixel_hash=hashlib.sha256(str(arr.shape).encode()+arr.tobytes()).hexdigest()
            manifest.append(dict(image_id=r.image_id,group_id=r.group_id,source=r.source,split=r.split,
                                 file=stem,brightness_factor=factor,pixel_hash=pixel_hash))
        if i%100==0:print(scenario,i,'/252')
    manifest=pd.DataFrame(manifest)
    assert manifest.groupby('image_id').split.nunique().max()==1
    assert manifest.groupby('pixel_hash').split.nunique().max()==1
    assert (manifest.loc[manifest.split!='train','brightness_factor']==1).all()
    expected=602 if setting['augment'] else 252
    assert len(manifest)==expected
    manifest.to_csv(ABLATION_HOME/f'{scenario}_manifest.csv',index=False)
    config=dict(path=str(root),train='train/images',val='valid/images',test='test/images',nc=20,names=FDI)
    (root/'data.yaml').write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    DATASETS[scenario]=root
    for split,g in manifest.groupby('split'):
        dataset_rows.append(dict(scenario=scenario,split=split,citra_asli=g.image_id.nunique(),file=len(g)))
display(pd.DataFrame(dataset_rows))
pd.DataFrame(dataset_rows).to_csv(ABLATION_HOME/'deskripsi_dataset_ablasi.csv',index=False)


A0_asli_bersih 100 /252
A0_asli_bersih 200 /252


,scenario,split,citra_asli,file
0,A0_asli_bersih,test,27,27
1,A0_asli_bersih,train,175,175
2,A0_asli_bersih,valid,50,50


In [5]:
"""Baseline YOLO11n vs YOLO11m. Training dipanggil di Colab GPU, bukan saat import."""
from pathlib import Path
import json,time
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

TRAIN_CONFIG=dict(imgsz=640,batch=8,epochs=80,patience=15,optimizer='AdamW',lr0=0.001,
                  weight_decay=0.0005,cos_lr=True,seed=42,deterministic=True,workers=2,
                  amp=True,pretrained=True,plots=True,save=True,val=True,cache=False,
                  mosaic=0.0,mixup=0.0,copy_paste=0.0,degrees=0.0,translate=0.0,scale=0.0,
                  shear=0.0,perspective=0.0,flipud=0.0,fliplr=0.0,
                  hsv_h=0.0,hsv_s=0.0,hsv_v=0.0,bgr=0.0)

def train_one(weights,data_yaml,runs_dir,name):
    import torch,ultralytics
    from ultralytics import YOLO
    if not torch.cuda.is_available():
        raise RuntimeError('Aktifkan GPU Colab: Runtime > Change runtime type > T4 GPU. Training CPU tidak dimulai otomatis.')
    model=YOLO(weights);start=time.time()
    # Train memakai validation, tidak memakai test untuk early stopping.
    model.train(data=str(data_yaml),device=0,project=str(runs_dir),name=name,exist_ok=False,**TRAIN_CONFIG)
    save_dir=Path(model.trainer.save_dir);best=save_dir/'weights'/'best.pt'
    if not best.is_file():raise RuntimeError('Training selesai tetapi best.pt tidak ditemukan.')
    hist=pd.read_csv(save_dir/'results.csv');hist.columns=hist.columns.str.strip()
    info=dict(model=weights,best=str(best),save_dir=str(save_dir),epochs_completed=len(hist),
              seconds=time.time()-start,ultralytics=ultralytics.__version__,torch=torch.__version__,
              gpu=torch.cuda.get_device_name(0),training_config=TRAIN_CONFIG)
    (save_dir/'run_metadata.json').write_text(json.dumps(info,indent=2),encoding='utf-8')
    del model
    torch.cuda.empty_cache()
    return info

def box_iou(gt,pred):
    gt=np.asarray(gt,dtype=float).reshape(-1,4);pred=np.asarray(pred,dtype=float).reshape(-1,4)
    if not len(gt) or not len(pred):return np.zeros((len(gt),len(pred)))
    lt=np.maximum(gt[:,None,:2],pred[None,:,:2]);rb=np.minimum(gt[:,None,2:],pred[None,:,2:])
    inter=np.maximum(rb-lt,0).prod(axis=2)
    area1=np.maximum(gt[:,2:]-gt[:,:2],0).prod(axis=1)
    area2=np.maximum(pred[:,2:]-pred[:,:2],0).prod(axis=1)
    return inter/np.maximum(area1[:,None]+area2[None,:]-inter,1e-12)

def match_boxes(gt,pred,iou_threshold=.5):
    ious=box_iou(gt,pred)
    if not ious.size:return []
    valid=ious>=iou_threshold
    # Maksimalkan jumlah pasangan valid dahulu, baru total IoU.
    score=valid.astype(float)+ious/(max(ious.shape)+1)
    rows,cols=linear_sum_assignment(score,maximize=True)
    return [(int(g),int(p)) for g,p in zip(rows,cols) if valid[g,p]]

def nominal_kappa(true,pred,num_classes=21):
    true=np.asarray(true,int);pred=np.asarray(pred,int)
    if not len(true):return float('nan')
    cm=np.zeros((num_classes,num_classes),int);np.add.at(cm,(true,pred),1)
    observed=np.trace(cm)/cm.sum()
    expected=(cm.sum(0)@cm.sum(1))/float(cm.sum()**2)
    return float((observed-expected)/(1-expected)) if not np.isclose(expected,1) else float('nan')

def evaluate_cache(records,confidence=.25,names=None,iou_threshold=.5):
    names=names or [str(i) for i in range(20)]
    k=len(names);cm=np.zeros((k+1,k+1),int);true_units=[];pred_units=[]
    image_rows=[]
    for r in records:
        keep=np.asarray(r['pred_conf'])>=confidence
        pb=np.asarray(r['pred_boxes']).reshape(-1,4)[keep]
        pc=np.asarray(r['pred_classes'],int)[keep]
        gb=np.asarray(r['gt_boxes']).reshape(-1,4);gc=np.asarray(r['gt_classes'],int)
        pairs=match_boxes(gb,pb,iou_threshold);used_g=set();used_p=set()
        for g,p in pairs:
            cm[gc[g],pc[p]]+=1;true_units.append(int(gc[g]));pred_units.append(int(pc[p]))
            used_g.add(g);used_p.add(p)
        for g in range(len(gc)):
            if g not in used_g:
                cm[gc[g],k]+=1;true_units.append(int(gc[g]));pred_units.append(k)
        for p in range(len(pc)):
            if p not in used_p:cm[k,pc[p]]+=1
        image_rows.append(dict(filename=r['filename'],gt=len(gc),pred=len(pc),matched=len(pairs)))
    tp=np.diag(cm)[:k];fp=cm.sum(0)[:k]-tp;fn=cm.sum(1)[:k]-tp
    precision=np.divide(tp,tp+fp,out=np.zeros(k,float),where=(tp+fp)>0)
    recall=np.divide(tp,tp+fn,out=np.zeros(k,float),where=(tp+fn)>0)
    f1=np.divide(2*tp,2*tp+fp+fn,out=np.zeros(k,float),where=(2*tp+fp+fn)>0)
    totals=dict(tp=int(tp.sum()),fp=int(fp.sum()),fn=int(fn.sum()))
    totals['precision_micro']=totals['tp']/max(1,totals['tp']+totals['fp'])
    totals['recall_micro']=totals['tp']/max(1,totals['tp']+totals['fn'])
    totals['f1_micro']=2*totals['tp']/max(1,2*totals['tp']+totals['fp']+totals['fn'])
    evaluated=(tp+fn)>0
    totals['f1_macro_present_classes']=float(f1[evaluated].mean()) if evaluated.any() else float('nan')
    totals['kappa_gt_units']=nominal_kappa(true_units,pred_units,k+1)
    totals['confidence_from_validation']=confidence;totals['match_iou']=iou_threshold
    table=pd.DataFrame(dict(FDI=names,TP=tp,FP=fp,FN=fn,precision=precision,recall=recall,F1=f1))
    return totals,table,cm,pd.DataFrame(image_rows)

def predict_cache(weights,dataset_root,split,device=0):
    from ultralytics import YOLO
    from PIL import Image
    root=Path(dataset_root);paths=sorted((root/split/'images').glob('*.png'))
    if not paths:raise FileNotFoundError(root/split/'images')
    model=YOLO(str(weights));records=[]
    stream=model.predict(source=list(map(str,paths)),stream=True,imgsz=640,batch=8,device=device,
                         conf=.001,iou=.7,max_det=300,verbose=False)
    for result in stream:
        p=Path(result.path);h,w=result.orig_shape;gt=[];classes=[]
        label=root/split/'labels'/(p.stem+'.txt')
        for line in label.read_text().splitlines():
            if not line.strip():continue
            c,x,y,bw,bh=map(float,line.split());classes.append(int(c))
            gt.append([(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h])
        boxes=result.boxes
        records.append(dict(filename=p.name,gt_boxes=np.array(gt).reshape(-1,4),gt_classes=np.array(classes,int),
                            pred_boxes=boxes.xyxy.cpu().numpy(),pred_classes=boxes.cls.cpu().numpy().astype(int),
                            pred_conf=boxes.conf.cpu().numpy()))
    return records

def choose_threshold(weights,dataset_root,output_dir,names):
    # Pilih threshold dengan validation saja, kemudian kunci sebelum test.
    records=predict_cache(weights,dataset_root,'valid');rows=[]
    for threshold in np.arange(.05,.96,.05):
        metrics,_,_,_=evaluate_cache(records,float(threshold),names)
        rows.append(metrics)
    frame=pd.DataFrame(rows)
    frame.to_csv(Path(output_dir)/'threshold_validation.csv',index=False)
    best=frame.sort_values(['f1_micro','confidence_from_validation'],ascending=[False,True]).iloc[0]
    return float(best.confidence_from_validation)

def test_model(weights,data_yaml,dataset_root,output_dir,confidence,names):
    from ultralytics import YOLO
    import matplotlib.pyplot as plt
    output_dir=Path(output_dir);output_dir.mkdir(exist_ok=True,parents=True)
    model=YOLO(str(weights))
    result=model.val(data=str(data_yaml),split='test',imgsz=640,batch=8,device=0,
                     conf=.001,iou=.7,plots=True,project=str(output_dir),name='ultralytics_test',exist_ok=False)
    cache=predict_cache(weights,dataset_root,'test')
    summary,per_class,cm,per_image=evaluate_cache(cache,confidence,names)
    summary.update(map50=float(result.box.map50),map50_95=float(result.box.map))
    # AP untuk kelas tidak tersedia tidak diisi dengan rata-rata keseluruhan.
    present=set(map(int,result.box.ap_class_index))
    per_class['AP50_95']=[float(result.box.maps[i]) if i in present else np.nan for i in range(len(names))]
    per_class.to_csv(output_dir/'test_per_FDI.csv',index=False)
    per_image.to_csv(output_dir/'test_per_image.csv',index=False)
    pd.DataFrame(cm,index=names+['background'],columns=names+['background']).to_csv(output_dir/'confusion_matrix_gt_rows.csv')
    (output_dir/'test_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    fig,ax=plt.subplots(figsize=(10,8));im=ax.imshow(cm,cmap='Blues')
    ax.set_xticks(range(len(names)+1),names+['BG'],rotation=90)
    ax.set_yticks(range(len(names)+1),names+['BG'])
    ax.set(xlabel='Prediksi',ylabel='Acuan',title=f'Confusion matrix — confidence validation {confidence:.2f}, IoU match 0.5')
    fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(output_dir/'confusion_matrix_gt_rows.png',dpi=160);plt.close(fig)
    return summary

assert TRAIN_CONFIG['patience']==15
for key,value in TRAIN_CONFIG.items():
    assert baseline_meta['training_config'][key]==value, f'Konfigurasi baseline berbeda pada {key}'


## Training ulang YOLO11n citra asli — patience 15

In [6]:
RAW_MODEL=train_one('yolo11n.pt',DATASETS['A0_asli_bersih']/'data.yaml',ABLATION_HOME,'asli_bersih_patience15')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.194 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/fdi_ablasi_20261031_173820/A0_asli_bersih/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, for

## Pilih confidence pada validation dan evaluasi test

In [7]:
evaluation=ABLATION_HOME/'asli_bersih_patience15'/'evaluasi'
evaluation.mkdir(parents=True,exist_ok=True)
threshold=choose_threshold(RAW_MODEL['best'],DATASETS['A0_asli_bersih'],evaluation,FDI)
raw_metrics=test_model(RAW_MODEL['best'],DATASETS['A0_asli_bersih']/'data.yaml',DATASETS['A0_asli_bersih'],evaluation,threshold,FDI)
comparison=pd.DataFrame([
 dict(kondisi='Citra asli bersih tanpa HE dan augmentasi',patience=15,epochs_completed=RAW_MODEL['epochs_completed'],**raw_metrics),
 dict(kondisi='Baseline YOLO11n dengan HE dan brightness',patience=15,epochs_completed=baseline_meta['epochs_completed'],**BASELINE_RESULT)
])
comparison.to_csv(ABLATION_HOME/'perbandingan_dua_kondisi.csv',index=False)
display(comparison[['kondisi','patience','epochs_completed','map50','map50_95','precision_micro','recall_micro','f1_micro','kappa_gt_units']])
delta=(float(BASELINE_RESULT['map50'])-raw_metrics['map50'])*100
interpretasi=f'Selisih mAP50 baseline HE + brightness terhadap citra asli bersih: {delta:+.2f} poin persen.'
print(interpretasi)
print('Positif berarti paket pengolahan lebih tinggi pada run ini, negatif berarti citra asli lebih tinggi. Satu run tidak membuktikan signifikansi atau generalisasi. Efek HE dan brightness tidak dipisahkan.')
report=pd.DataFrame({'Kondisi':comparison.kondisi,'Patience':comparison.patience,
 'mAP50 (%)':comparison.map50.mul(100),'mAP50-95 (%)':comparison.map50_95.mul(100),
 'F1 micro (%)':comparison.f1_micro.mul(100),'Kappa':comparison.kappa_gt_units})
report.to_csv(ABLATION_HOME/'tabel_singkat_untuk_laporan.csv',index=False)
(ABLATION_HOME/'ringkasan_untuk_laporan.txt').write_text(report.round(3).to_string(index=False)+'\n'+interpretasi,encoding='utf-8')
print('SELESAI. Tabel hasil untuk laporan:',ABLATION_HOME/'tabel_singkat_untuk_laporan.csv')


Ultralytics 8.3.221 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,586,052 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2811.7±270.1 MB/s, size: 1325.3 KB)
val: Scanning /content/fdi_ablasi_20261031_173820/A0_asli_bersih/test/labels... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 748.1it/s 0.0s
val: New cache created: /content/fdi_ablasi_20261031_173820/A0_asli_bersih/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.8it/s 2.2s
                   all         27        327      0.731      0.702       0.75      0.397
                    51          7          7      0.502      0.857      0.651       0.38
                    52         11         11      0.521      0.636      0.804      0.488
                    53         22         22      0.697      0.818      0.844      0.446
                 

,kondisi,patience,epochs_completed,map50,map50_95,precision_micro,recall_micro,f1_micro,kappa_gt_units
0,Citra asli bersih tanpa HE dan augmentasi,15,38,0.750110,0.397230,0.842308,0.669725,0.746167,0.655538
1,Baseline YOLO11n dengan HE dan brightness,15,59,0.788892,0.427146,0.840741,0.694190,0.760469,0.680074


Selisih mAP50 baseline HE + brightness terhadap citra asli bersih: +3.88 poin persen.
Positif berarti paket pengolahan lebih tinggi pada run ini, negatif berarti citra asli lebih tinggi. Satu run tidak membuktikan signifikansi atau generalisasi. Efek HE dan brightness tidak dipisahkan.
SELESAI. Tabel hasil untuk laporan: /content/drive/MyDrive/1226800011_Aprilia_UjianEDA_Revisi/hasil_perbandingan_dua_YOLO11n/20261031_173820/tabel_singkat_untuk_laporan.csv
